In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import re
import pandas as pd
import numpy as np

from tqdm import tqdm

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# CELL 2 — LOAD DATASET
# ============================================================

DATASET_PATH = "../../parquet_exports_v2/gold_ticket_similarity_v2.parquet"

df = pd.read_parquet(DATASET_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

df.head()

DATASET LOADED
Rows    : 108,536
Columns : 47


,rag_id,problem_text,solution_text,category,priority,language,product,queue,software_version,region,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,priority_tier,urgency_tier,impact_tier,triage_priority,escalation_risk_level,age_severity,customer_type,resolution_time_hours,waiting_duration,first_response_time_hours,sla_risk_score,escalation_probability,is_sla_breached,is_escalated,issue_complexity_score,customer_satisfaction_score,customer_tenure_months,previous_tickets,customer_segment,subscription_type,operating_system,browser,followup_count,avg_followup_content_length,source_dataset,created_at,resolved_at,source_system,source_id
0,45d0726cc395d2b3a2b8c9899523e1dc,I am unable to access my account after enterin...,Security settings updated and customer notifie...,performance issue,critical,french,Subscription Service,NaN,NaN,europe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,critical,None,None,medium,low,low,frequent,0.00,None,38.20,3.75,2.25,1,0,4.0,2.0,19.0,19.0,Individual,enterprise,linux,edge,0,0.0,customer_support_tickets_200k,2022-09-08,2022-09-08,customer_support_tickets_200k,383
1,504f35fd3b03f048a1984f89fffb08e1,I would like to request a refund for the recen...,Explained billing breakdown and clarified appl...,login issue,medium,french,E-commerce Store,NaN,NaN,north america,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,medium,None,None,medium,low,high,frequent,127.87,None,35.94,2.80,1.95,1,0,2.0,1.0,35.0,20.0,Corporate,free,macos,chrome,0,0.0,customer_support_tickets_200k,2023-09-25,2023-09-28,customer_support_tickets_200k,710
2,db7bc1e2f75fa986ee832c31e320a2de,There seems to be a discrepancy in my billing ...,Provided step-by-step troubleshooting instruct...,refund request,high,english,Cloud Storage,NaN,NaN,africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,high,None,None,high,low,low,frequent,20.94,None,18.89,4.00,2.60,0,0,7.0,1.0,46.0,18.0,Small Business,free,ios,safari,0,0.0,customer_support_tickets_200k,2023-06-21,2023-06-21,customer_support_tickets_200k,1031
3,01d7f2addbcc200f88368695230ea06e,The system is not syncing data across devices ...,Bug logged internally and workaround shared wi...,payment problem,low,chinese,Analytics Dashboard,NaN,NaN,north america,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low,None,None,medium,low,critical,new,63.57,None,14.06,2.95,3.25,0,1,6.0,2.0,54.0,0.0,Corporate,basic,linux,firefox,0,0.0,customer_support_tickets_200k,2023-03-13,2023-03-25,customer_support_tickets_200k,1114
4,aa219338e9e467aadb53d3d1a9026de7,I am unable to access my account after enterin...,Provided step-by-step troubleshooting instruct...,security concern,low,german,Cloud Storage,NaN,NaN,australia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low,None,None,medium,low,critical,frequent,159.59,None,36.16,3.30,3.25,1,1,7.0,5.0,29.0,19.0,Corporate,basic,linux,NaN,0,0.0,customer_support_tickets_200k,2024-10-12,2024-10-21,customer_support_tickets_200k,1616


In [3]:
# ============================================================
# CELL 3 — BUILD RETRIEVAL TEXT
# ============================================================

work_df = df.copy()

print("=" * 60)
print("TEXT NULL CHECK")
print("=" * 60)

null_problem = work_df["problem_text"].isna().sum()
null_solution = work_df["solution_text"].isna().sum()

print(f"Null problem_text  : {null_problem:,}")
print(f"Null solution_text : {null_solution:,}")

# ------------------------------------------------------------
# DROP ROWS WHERE EITHER IS NULL
# ------------------------------------------------------------

before_rows = len(work_df)

work_df = work_df[
    work_df["problem_text"].notna()
].copy()

work_df = work_df[
    work_df["solution_text"].notna()
].copy()

after_rows = len(work_df)

print("\nRows before null filter :", f"{before_rows:,}")
print("Rows after null filter  :", f"{after_rows:,}")
print("Removed null rows       :", f"{before_rows - after_rows:,}")

# ------------------------------------------------------------
# BUILD RETRIEVAL TEXT FOR BOTH COLUMNS
# ------------------------------------------------------------

work_df["problem_text"] = (
    work_df["problem_text"]
    .fillna("")
    .astype(str)
)

work_df["solution_text"] = (
    work_df["solution_text"]
    .fillna("")
    .astype(str)
)

# Remove empty texts
work_df = work_df[
    work_df["problem_text"].str.strip() != ""
].copy()

work_df = work_df[
    work_df["solution_text"].str.strip() != ""
].copy()

print("\nRetrieval text created successfully for problem_text and solution_text.")

print("\nSample problem_text:")
print(work_df["problem_text"].iloc[0][:500])

print("\nSample solution_text:")
print(work_df["solution_text"].iloc[0][:500])

TEXT NULL CHECK
Null problem_text  : 0
Null solution_text : 0

Rows before null filter : 108,536
Rows after null filter  : 108,536
Removed null rows       : 0

Retrieval text created successfully for problem_text and solution_text.

Sample problem_text:
I am unable to access my account after entering the correct credentials. I am unable to access my account after entering the correct credentials.

Sample solution_text:
Security settings updated and customer notified of precautionary measures.


In [4]:
print("=" * 60)
print("TEXT VALIDATION")
print("=" * 60)

before_rows = len(work_df)

work_df = work_df[
    work_df["problem_text"].notna()
].copy()

work_df = work_df[
    work_df["problem_text"].str.len() >= 20
].copy()

work_df = work_df[
    work_df["solution_text"].notna()
].copy()

work_df = work_df[
    work_df["solution_text"].str.len() >= 20
].copy()

print(f"Removed rows: {before_rows - len(work_df):,}")
print(f"Remaining   : {len(work_df):,}")

TEXT VALIDATION
Removed rows: 0
Remaining   : 108,536


In [5]:
# ============================================================
# CELL 4 — BASIC TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text: str) -> str:
    """
    Production-oriented text cleaner
    for ticket retrieval systems.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # lowercase
    text = text.lower()

    # remove html
    text = re.sub(r"<.*?>", " ", text)

    # remove urls
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove emails
    text = re.sub(r"\S+@\S+", " ", text)

    # remove ip addresses
    text = re.sub(
        r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b",
        " ",
        text
    )

    # remove ticket ids
    text = re.sub(
        r"(ticket|inc|req)[-_ ]?\d+",
        " ",
        text
    )

    # remove repeated punctuation
    text = re.sub(r"([!?.,])\1+", r"\1", text)

    # remove extra symbols
    text = re.sub(r"[^a-zA-Z0-9\s\-_./]", " ", text)

    # remove line breaks
    text = re.sub(r"[\n\r\t]", " ", text)

    # remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("clean_text function created.")

clean_text function created.


In [6]:
# ============================================================
# CELL 5 — REMOVE COMMON TICKET NOISE
# ============================================================

NOISE_PATTERNS = [
    r"best regards",
    r"kind regards",
    r"thanks regards",
    r"thank you",
    r"sent from my iphone",
    r"please help",
    r"dear support",
    r"hello team",
    r"hi team",
    r"good morning",
    r"good afternoon",
]

def remove_ticket_noise(text: str) -> str:

    if pd.isna(text):
        return ""

    text = str(text)

    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("Noise removal function created.")

Noise removal function created.


In [7]:
# ============================================================
# CELL 6 — REMOVE REPEATED PHRASES
# ============================================================

def remove_repeated_phrases(text: str) -> str:

    if pd.isna(text):
        return ""

    text = str(text)

    # remove duplicated consecutive phrases
    text = re.sub(
        r"\b(.+?)(?:\s+\1\b)+",
        r"\1",
        text
    )

    return text.strip()


print("Repeated phrase cleaner created.")

Repeated phrase cleaner created.


In [8]:
# ============================================================
# CELL 7 — APPLY CLEANING PIPELINE
# ============================================================

tqdm.pandas(desc="Cleaning problem_text")

work_df["problem_text_clean"] = (
    work_df["problem_text"]
    .progress_apply(clean_text)
    .progress_apply(remove_ticket_noise)
    .progress_apply(remove_repeated_phrases)
)

tqdm.pandas(desc="Cleaning solution_text")

work_df["solution_text_clean"] = (
    work_df["solution_text"]
    .progress_apply(clean_text)
    .progress_apply(remove_ticket_noise)
    .progress_apply(remove_repeated_phrases)
)

print("Cleaning pipeline applied successfully.")

work_df[
    [
        "problem_text",
        "problem_text_clean",
        "solution_text",
        "solution_text_clean"
    ]
].head()

Cleaning solution_text: 100%|██████████| 108536/108536 [01:29<00:00, 1217.85it/s]

Cleaning pipeline applied successfully.


,problem_text,problem_text_clean,solution_text,solution_text_clean
0,I am unable to access my account after enterin...,i am unable to access my account after enterin...,Security settings updated and customer notifie...,security settings updated and customer notifie...
1,I would like to request a refund for the recen...,i would like to request a refund for the recen...,Explained billing breakdown and clarified appl...,explained billing breakdown and clarified appl...
2,There seems to be a discrepancy in my billing ...,there seems to be a discrepancy in my billing ...,Provided step-by-step troubleshooting instruct...,provided step-by-step troubleshooting instruct...
3,The system is not syncing data across devices ...,the system is not syncing data across devices ...,Bug logged internally and workaround shared wi...,bug logged internally and workaround shared wi...
4,I am unable to access my account after enterin...,i am unable to access my account after enterin...,Provided step-by-step troubleshooting instruct...,provided step-by-step troubleshooting instruct...


In [9]:
before_rows = len(work_df)

work_df = work_df[
    work_df["problem_text_clean"].notna()
].copy()

work_df = work_df[
    work_df["problem_text_clean"].str.strip() != ""
].copy()

work_df = work_df[
    work_df["problem_text_clean"].str.len() >= 20
].copy()

work_df = work_df[
    work_df["solution_text_clean"].notna()
].copy()

work_df = work_df[
    work_df["solution_text_clean"].str.strip() != ""
].copy()

work_df = work_df[
    work_df["solution_text_clean"].str.len() >= 20
].copy()

removed_rows = before_rows - len(work_df)

print("=" * 60)
print("EMPTY / SHORT TEXT FILTER")
print("=" * 60)

print(f"Removed rows : {removed_rows:,}")
print(f"Remaining    : {len(work_df):,}")

EMPTY / SHORT TEXT FILTER
Removed rows : 2
Remaining    : 108,534


In [10]:
# ============================================================
# CELL 9 — TEXT LENGTH FEATURES
# ============================================================

work_df["problem_word_count"] = (
    work_df["problem_text_clean"]
    .str.split()
    .apply(len)
)

work_df["problem_char_count"] = (
    work_df["problem_text_clean"]
    .str.len()
)

work_df["solution_word_count"] = (
    work_df["solution_text_clean"]
    .str.split()
    .apply(len)
)

work_df["solution_char_count"] = (
    work_df["solution_text_clean"]
    .str.len()
)

print("Text length features created.")

work_df[
    [
        "problem_word_count",
        "problem_char_count",
        "solution_word_count",
        "solution_char_count"
    ]
].describe()

Text length features created.


,problem_word_count,problem_char_count,solution_word_count,solution_char_count
count,108534.000000,108534.000000,108534.000000,108534.000000
mean,32.160208,208.800127,22.099628,149.787081
std,22.837652,163.159971,25.632556,163.281195
min,2.000000,23.000000,2.000000,20.000000
25%,20.000000,127.000000,8.000000,63.000000
50%,22.000000,143.000000,9.000000,72.000000
75%,26.000000,159.000000,16.000000,104.000000
max,201.000000,1512.000000,136.000000,969.000000


In [11]:
# ============================================================
# CELL 10 — REMOVE VERY SHORT TICKETS
# ============================================================

MIN_WORDS = 5

before_filter = len(work_df)

work_df = work_df[
    work_df["problem_word_count"] >= MIN_WORDS
].copy()

work_df = work_df[
    work_df["solution_word_count"] >= MIN_WORDS
].copy()

removed_short = before_filter - len(work_df)

print("=" * 60)
print("SHORT TICKET FILTER")
print("=" * 60)

print(f"Removed short tickets : {removed_short:,}")
print(f"Remaining tickets     : {len(work_df):,}")

SHORT TICKET FILTER
Removed short tickets : 113
Remaining tickets     : 108,421


In [12]:
# ============================================================
# CELL 11 — REMOVE DUPLICATED TICKETS
# ============================================================

before_duplicates = len(work_df)

if "rag_id" in work_df.columns:

    work_df = (
        work_df
        .drop_duplicates(subset=["rag_id"])
        .reset_index(drop=True)
    )

removed_duplicates = before_duplicates - len(work_df)

print("=" * 60)
print("DUPLICATE REMOVAL")
print("=" * 60)

print(f"Removed duplicates : {removed_duplicates:,}")
print(f"Remaining tickets  : {len(work_df):,}")

DUPLICATE REMOVAL
Removed duplicates : 0
Remaining tickets  : 108,421


In [13]:
# ============================================================
# CELL 12 — REPETITION RATIO ANALYSIS
# ============================================================

def repetition_ratio(text: str) -> float:

    words = str(text).split()

    if len(words) == 0:
        return 0.0

    return 1 - (len(set(words)) / len(words))


work_df["problem_repetition_ratio"] = (
    work_df["problem_text_clean"]
    .apply(repetition_ratio)
)

work_df["solution_repetition_ratio"] = (
    work_df["solution_text_clean"]
    .apply(repetition_ratio)
)

print("Repetition ratios created.")

work_df[
    [
        "problem_repetition_ratio",
        "solution_repetition_ratio"
    ]
].describe()

Repetition ratios created.


,problem_repetition_ratio,solution_repetition_ratio
count,108421.000000,108421.000000
mean,0.421412,0.055289
std,0.152089,0.082525
min,0.000000,0.000000
25%,0.298077,0.000000
50%,0.500000,0.000000
75%,0.500000,0.083333
max,0.545455,0.419355


In [14]:
# ============================================================
# CELL 13 — REMOVE HIGHLY REPETITIVE TICKETS
# ============================================================

MAX_REPETITION_RATIO = 0.60

before_repetition = len(work_df)

work_df = work_df[
    work_df["problem_repetition_ratio"] <= MAX_REPETITION_RATIO
].copy()

work_df = work_df[
    work_df["solution_repetition_ratio"] <= MAX_REPETITION_RATIO
].copy()

removed_repetition = before_repetition - len(work_df)

print("=" * 60)
print("REPETITION FILTER")
print("=" * 60)

print(f"Removed repetitive tickets : {removed_repetition:,}")
print(f"Remaining tickets          : {len(work_df):,}")

REPETITION FILTER
Removed repetitive tickets : 0
Remaining tickets          : 108,421


In [15]:
# ============================================================
# CELL 14 — FINAL RETRIEVAL DATAFRAME
# ============================================================

REQUIRED_COLUMNS = [
    "rag_id",
    "problem_text_clean",
    "solution_text_clean",
    "category",
    "priority",
    "language",
    "product",
    "queue",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4",
    "tag_5",
    "tag_6",
    "tag_7",
    "tag_8",
    "is_sla_breached",
    "is_escalated",
    "escalation_risk_level",
    "issue_complexity_score",
    "sla_risk_score",
    "problem_word_count",
    "problem_char_count",
    "problem_repetition_ratio",
    "solution_word_count",
    "solution_char_count",
    "solution_repetition_ratio",
    "source_system",
    "source_dataset",
    "created_at",
    "resolved_at"
]

existing_columns = [
    c for c in REQUIRED_COLUMNS
    if c in work_df.columns
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in work_df.columns
]

print("=" * 60)
print("COLUMN VALIDATION")
print("=" * 60)

print("Existing columns:")
print(existing_columns)

print("\nMissing columns:")
print(missing_columns)

retrieval_df = (
    work_df[existing_columns]
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL RETRIEVAL DATAFRAME")
print("=" * 60)

print(f"Rows    : {len(retrieval_df):,}")
print(f"Columns : {retrieval_df.shape[1]}")

retrieval_df.head()

COLUMN VALIDATION
Existing columns:
['rag_id', 'problem_text_clean', 'solution_text_clean', 'category', 'priority', 'language', 'product', 'queue', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'is_sla_breached', 'is_escalated', 'escalation_risk_level', 'issue_complexity_score', 'sla_risk_score', 'problem_word_count', 'problem_char_count', 'problem_repetition_ratio', 'solution_word_count', 'solution_char_count', 'solution_repetition_ratio', 'source_system', 'source_dataset', 'created_at', 'resolved_at']

Missing columns:
[]
FINAL RETRIEVAL DATAFRAME
Rows    : 108,421
Columns : 31


,rag_id,problem_text_clean,solution_text_clean,category,priority,language,product,queue,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,is_sla_breached,is_escalated,escalation_risk_level,issue_complexity_score,sla_risk_score,problem_word_count,problem_char_count,problem_repetition_ratio,solution_word_count,solution_char_count,solution_repetition_ratio,source_system,source_dataset,created_at,resolved_at
0,45d0726cc395d2b3a2b8c9899523e1dc,i am unable to access my account after enterin...,security settings updated and customer notifie...,performance issue,critical,french,Subscription Service,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,low,4.0,3.75,24,145,0.5,9,74,0.0,customer_support_tickets_200k,customer_support_tickets_200k,2022-09-08,2022-09-08
1,504f35fd3b03f048a1984f89fffb08e1,i would like to request a refund for the recen...,explained billing breakdown and clarified appl...,login issue,medium,french,E-commerce Store,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,low,2.0,2.80,22,111,0.5,7,61,0.0,customer_support_tickets_200k,customer_support_tickets_200k,2023-09-25,2023-09-28
2,db7bc1e2f75fa986ee832c31e320a2de,there seems to be a discrepancy in my billing ...,provided step-by-step troubleshooting instruct...,refund request,high,english,Cloud Storage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,low,7.0,4.00,26,143,0.5,7,70,0.0,customer_support_tickets_200k,customer_support_tickets_200k,2023-06-21,2023-06-21
3,01d7f2addbcc200f88368695230ea06e,the system is not syncing data across devices ...,bug logged internally and workaround shared wi...,payment problem,low,chinese,Analytics Dashboard,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,low,6.0,2.95,18,111,0.5,9,62,0.0,customer_support_tickets_200k,customer_support_tickets_200k,2023-03-13,2023-03-25
4,aa219338e9e467aadb53d3d1a9026de7,i am unable to access my account after enterin...,provided step-by-step troubleshooting instruct...,security concern,low,german,Cloud Storage,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,low,7.0,3.30,24,145,0.5,7,70,0.0,customer_support_tickets_200k,customer_support_tickets_200k,2024-10-12,2024-10-21


In [16]:
# ============================================================
# CELL 15 — FINAL DATASET HEALTH CHECK
# ============================================================

print("=" * 60)
print("FINAL DATASET HEALTH CHECK")
print("=" * 60)

print(f"Final Rows                     : {len(retrieval_df):,}")
print(f"Average Problem Word Count     : {retrieval_df['problem_word_count'].mean():.2f}")
print(f"Average Problem Char Count     : {retrieval_df['problem_char_count'].mean():.2f}")
print(f"Average Problem Repetition     : {retrieval_df['problem_repetition_ratio'].mean():.4f}")
print(f"Average Solution Word Count    : {retrieval_df['solution_word_count'].mean():.2f}")
print(f"Average Solution Char Count    : {retrieval_df['solution_char_count'].mean():.2f}")
print(f"Average Solution Repetition    : {retrieval_df['solution_repetition_ratio'].mean():.4f}")

if "source_system" in retrieval_df.columns:

    print("\nSource Systems:")
    print(
        retrieval_df["source_system"]
        .value_counts()
    )

FINAL DATASET HEALTH CHECK
Final Rows                     : 108,421
Average Problem Word Count     : 32.15
Average Problem Char Count     : 208.74
Average Problem Repetition     : 0.4214
Average Solution Word Count    : 22.11
Average Solution Char Count    : 149.88
Average Solution Repetition    : 0.0553

Source Systems:
source_system
customer_support_tickets_200k    79999
dataset_tickets_multi_lang       28422
Name: count, dtype: int64


In [17]:
# ============================================================
# CELL 16 — EXPORT CLEAN RETRIEVAL DATASET
# ============================================================

EXPORT_PATH = "../../parquet_exports_v2/retrieval_clean_ready_v2.parquet"

retrieval_df.to_parquet(
    EXPORT_PATH,
    index=False
)

print("=" * 60)
print("EXPORT COMPLETED")
print("=" * 60)

print(f"Exported dataset path:\n{EXPORT_PATH}")

EXPORT COMPLETED
Exported dataset path:
../../parquet_exports_v2/retrieval_clean_ready_v2.parquet


In [18]:
# ============================================================
# CELL 17 — EXPORT SAMPLE CSV
# ============================================================

SAMPLE_EXPORT = "../../evaluation_v2/retrieval_ready_v2_sample.csv"

retrieval_df.head(100).to_csv(
    SAMPLE_EXPORT,
    index=False
)

print(f"Sample exported to:\n{SAMPLE_EXPORT}")

Sample exported to:
../../evaluation_v2/retrieval_ready_v2_sample.csv
